# HW3 Image Classification — V3
## 新增：RandomErasing / MixUp / 更長訓練(80 epoch) / 更多TTA(16次) / 更大等效batch(256)
> Kaggle 上執行

# Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.is_available())

# Data Path

In [ ]:
import os
DATA_DIR = "/kaggle/input/competitions/ml2023spring-hw3"
print(os.listdir(DATA_DIR))

In [ ]:
_exp_name = "advanced_v3"

# Import Packages

In [ ]:
import numpy as np
import pandas as pd
import torch
import os
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder, VisionDataset
from tqdm.auto import tqdm
import random
import math

In [ ]:
myseed = 6666
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

# Transforms

**V3 新增：**
- `RandomErasing`：隨機遮住一塊區域，讓模型不依賴局部特徵
- `tta_tfm` 與 `train_tfm` 共用（不含 RandomErasing，避免 TTA 太激進）

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 測試 / 驗證用（固定，不隨機）
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# TTA 用（隨機增強，但不含 RandomErasing）
tta_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomCrop(128, padding=16),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# 訓練用（加入 RandomErasing，比 TTA 更激進）
# RandomErasing 必須在 ToTensor 之後（作用在 tensor 上）
train_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),       # 增強1：水平翻轉
    transforms.RandomVerticalFlip(p=0.2),          # 增強2：垂直翻轉
    transforms.RandomRotation(degrees=30),          # 增強3：旋轉
    transforms.ColorJitter(                         # 增強4：顏色抖動
        brightness=0.3, contrast=0.3,
        saturation=0.3, hue=0.1
    ),
    transforms.RandomCrop(128, padding=16),         # 增強5：隨機裁切
    transforms.RandomGrayscale(p=0.1),              # 增強6：灰階
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    # 增強7（新增）：隨機遮住 2%~20% 的區域
    # 強迫模型看整體而非局部特徵（例如不能只靠顏色區塊判斷）
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

# Dataset

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, path, tfm=test_tfm, files=None):
        super(FoodDataset).__init__()
        self.path = path
        self.files = sorted([
            os.path.join(path, x)
            for x in os.listdir(path)
            if x.endswith(".jpg")
        ])
        if files is not None:
            self.files = files
        self.transform = tfm

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        im = Image.open(fname)
        im = self.transform(im)
        try:
            label = int(os.path.basename(fname).split("_")[0])
        except:
            label = -1
        return im, label

# Model（Scale-up 版，同 V2）

6 個 Block，輸出 512×2×2，FC：2048→1024→512→11

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.cnn = nn.Sequential(
            # Block 1: [B, 3, 128, 128] → [B, 64, 64, 64]
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: [B, 64, 64, 64] → [B, 128, 32, 32]
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: [B, 128, 32, 32] → [B, 256, 16, 16]  ← mid layer (index=8)
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 4: [B, 256, 16, 16] → [B, 512, 8, 8]
            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 5: [B, 512, 8, 8] → [B, 512, 4, 4]
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 6: [B, 512, 4, 4] → [B, 512, 2, 2]  ← top layer (index=19)
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.fc = nn.Sequential(
            nn.Linear(512 * 2 * 2, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, 11),
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size(0), -1)
        return self.fc(out)

# Configurations

**V3 改動：**
- `n_epochs`: 50 → **80**
- `patience`: 10 → **15**
- `accumulation_steps`: 4 → **8**（等效 batch 32×8=**256**）
- `tta_times`: 8 → **16**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{device}")

batch_size         = 32
accumulation_steps = 8     # 等效 batch size = 32 × 8 = 256
n_epochs           = 80
patience           = 15
warmup_epochs      = 5
lr_max             = 3e-4
n_folds            = 5
tta_times          = 16    # TTA 次數加倍

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f"等效 batch size：{batch_size * accumulation_steps}")
print(f"n_epochs：{n_epochs}，patience：{patience}")
print(f"TTA 次數：{tta_times}")

# LR Warmup + Cosine Annealing（同 V2）

In [ ]:
def get_lr(epoch, warmup_epochs, n_epochs, lr_max, eta_min=1e-6):
    if epoch < warmup_epochs:
        return lr_max * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (n_epochs - warmup_epochs)
        return eta_min + 0.5 * (lr_max - eta_min) * (1 + math.cos(math.pi * progress))

# MixUp

**MixUp 的概念：**
把兩張圖按比例混合，標籤也跟著混合，是最強的資料增強方式之一。

```
混合圖 = λ × 圖A + (1-λ) × 圖B
loss   = λ × loss(預測, 標籤A) + (1-λ) × loss(預測, 標籤B)
```

讓模型學到圖片之間的連續過渡，而不是只學離散的分類邊界。

In [ ]:
def mixup_data(imgs, labels, alpha=0.2):
    """
    MixUp 資料增強。
    alpha：Beta 分布參數，越大混合越強（0.2 是常見預設值）
    回傳：混合後的圖片、兩個標籤、混合比例 lam
    """
    lam   = np.random.beta(alpha, alpha)  # 從 Beta 分布取混合比例
    index = torch.randperm(imgs.size(0)).to(imgs.device)  # 隨機配對 index

    mixed_imgs = lam * imgs + (1 - lam) * imgs[index]
    labels_a   = labels
    labels_b   = labels[index]
    return mixed_imgs, labels_a, labels_b, lam

def mixup_criterion(criterion, logits, labels_a, labels_b, lam):
    """MixUp 的 loss = 兩個標籤的 loss 按 lambda 加權"""
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)

# CV 資料切分

In [ ]:
all_train_files = (
    sorted([os.path.join(DATA_DIR, "train", x)
            for x in os.listdir(os.path.join(DATA_DIR, "train"))
            if x.endswith(".jpg")])
  + sorted([os.path.join(DATA_DIR, "valid", x)
            for x in os.listdir(os.path.join(DATA_DIR, "valid"))
            if x.endswith(".jpg")])
)
print(f"合併後總筆數：{len(all_train_files)}")

fold_size    = len(all_train_files) // n_folds
fold_indices = []
for k in range(n_folds):
    start = k * fold_size
    end   = start + fold_size if k < n_folds - 1 else len(all_train_files)
    fold_indices.append(list(range(start, end)))
print(f"每折大小約：{fold_size} 筆")

# Training Function（含 MixUp + Resume）

In [ ]:
def train_one_fold(fold_k, train_files, valid_files, resume_ckpt=None):
    """
    訓練單一折，回傳該折最佳 valid accuracy。
    V3 新增：MixUp 資料增強
    """
    print(f"\n{'='*50}")
    print(f"Fold {fold_k+1}/{n_folds}  訓練:{len(train_files)}  驗證:{len(valid_files)}")
    print(f"{'='*50}")

    train_set = FoodDataset(DATA_DIR+"/train", tfm=train_tfm, files=train_files)
    valid_set = FoodDataset(DATA_DIR+"/train", tfm=test_tfm,  files=valid_files)
    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=2,
                              persistent_workers=True, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size,
                              shuffle=False, num_workers=2,
                              persistent_workers=True, pin_memory=True)

    model     = nn.DataParallel(Classifier()).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_max, weight_decay=1e-5)

    start_epoch = 0
    best_acc    = 0
    stale       = 0

    if resume_ckpt is not None:
        model.module.load_state_dict(resume_ckpt["model"])
        optimizer.load_state_dict(resume_ckpt["optimizer"])
        start_epoch = resume_ckpt["epoch"] + 1
        best_acc    = resume_ckpt["best_acc"]
        stale       = resume_ckpt["stale"]
        print(f"從 epoch {start_epoch+1} 繼續，目前最佳 acc = {best_acc:.5f}")

    for epoch in range(start_epoch, n_epochs):
        lr = get_lr(epoch, warmup_epochs, n_epochs, lr_max)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # ────────── Training（含 MixUp）──────────
        model.train()
        train_loss, train_accs = [], []
        optimizer.zero_grad()

        for i, (imgs, labels) in enumerate(tqdm(train_loader, desc=f"Fold{fold_k+1} Epoch{epoch+1} Train")):
            imgs   = imgs.to(device)
            labels = labels.to(device)

            # MixUp：隨機混合兩張圖和標籤
            mixed_imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.2)

            logits = model(mixed_imgs)

            # MixUp loss：兩個標籤的加權組合
            loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)

            # Gradient Accumulation
            loss = loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
                optimizer.step()
                optimizer.zero_grad()

            # 準確率用 labels_a 計算（MixUp 後沒有單一正確答案）
            acc = (logits.argmax(dim=-1) == labels_a).float().mean()
            train_loss.append(loss.item() * accumulation_steps)
            train_accs.append(acc)

        train_loss = sum(train_loss) / len(train_loss)
        train_acc  = sum(train_accs) / len(train_accs)
        print(f"[ Train | {epoch+1:03d}/{n_epochs:03d} ] loss={train_loss:.4f} acc={train_acc:.4f} lr={lr:.2e}")

        # ────────── Validation（不用 MixUp）──────────
        model.eval()
        valid_loss, valid_accs = [], []

        for imgs, labels in tqdm(valid_loader, desc=f"Fold{fold_k+1} Epoch{epoch+1} Valid"):
            with torch.no_grad():
                logits = model(imgs.to(device))
            loss = criterion(logits, labels.to(device))
            acc  = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
            valid_loss.append(loss.item())
            valid_accs.append(acc)

        valid_loss = sum(valid_loss) / len(valid_loss)
        valid_acc  = sum(valid_accs) / len(valid_accs)
        print(f"[ Valid | {epoch+1:03d}/{n_epochs:03d} ] loss={valid_loss:.4f} acc={valid_acc:.4f}")

        with open(f"{_exp_name}_fold{fold_k}_log.txt", "a") as f:
            msg = f"[{epoch+1:03d}] loss={valid_loss:.4f} acc={valid_acc:.4f}"
            print(msg + (" -> best" if valid_acc > best_acc else ""), file=f)

        if valid_acc > best_acc:
            print(f"  ✓ Best model at epoch {epoch+1}, saving...")
            torch.save(model.module.state_dict(), f"{_exp_name}_fold{fold_k}.ckpt")
            best_acc = valid_acc
            stale = 0
        else:
            stale += 1
            if stale > patience:
                print(f"  Early stopping at epoch {epoch+1}")
                torch.save({
                    "fold":      fold_k,
                    "epoch":     epoch,
                    "model":     model.module.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "best_acc":  best_acc,
                    "stale":     stale,
                }, f"{_exp_name}_fold{fold_k}_checkpoint.pt")
                break

        # 每個 epoch 存完整 checkpoint（斷線可從這裡繼續）
        torch.save({
            "fold":      fold_k,
            "epoch":     epoch,
            "model":     model.module.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_acc":  best_acc,
            "stale":     stale,
        }, f"{_exp_name}_fold{fold_k}_checkpoint.pt")

    print(f"Fold {fold_k+1} 完成！最佳 valid acc = {best_acc:.5f}")
    return best_acc

# Run Cross Validation

In [ ]:
fold_accs = []

for k in range(n_folds):
    valid_idx = fold_indices[k]
    train_idx = []
    for j in range(n_folds):
        if j != k:
            train_idx.extend(fold_indices[j])
    train_files = [all_train_files[i] for i in train_idx]
    valid_files = [all_train_files[i] for i in valid_idx]

    ckpt_path        = f"{_exp_name}_fold{k}.ckpt"
    resume_ckpt_path = f"{_exp_name}_fold{k}_checkpoint.pt"

    if os.path.exists(ckpt_path) and not os.path.exists(resume_ckpt_path):
        print(f"Fold {k+1} 已完成，跳過")
        fold_accs.append(None)
        continue

    resume_ckpt = None
    if os.path.exists(resume_ckpt_path):
        print(f"Fold {k+1} 偵測到中斷點，從中繼續...")
        resume_ckpt = torch.load(resume_ckpt_path)

    acc = train_one_fold(k, train_files, valid_files, resume_ckpt)
    fold_accs.append(acc)

    if os.path.exists(resume_ckpt_path):
        os.remove(resume_ckpt_path)
        print(f"Fold {k+1} checkpoint 已清除")

print(f"\n{'='*50}")
print(f"Cross Validation 結果：")
for k, acc in enumerate(fold_accs):
    if acc is not None:
        print(f"  Fold {k+1}: {acc:.5f}")
    else:
        print(f"  Fold {k+1}: 已跳過")

valid_accs_list = [a for a in fold_accs if a is not None]
if valid_accs_list:
    print(f"  平均 valid acc: {sum(valid_accs_list)/len(valid_accs_list):.5f}")

# Testing with TTA（16 次）+ Ensemble

In [ ]:
test_set = FoodDataset(f"{DATA_DIR}/test", tfm=test_tfm)
test_loader_normal = DataLoader(test_set, batch_size=batch_size,
                                shuffle=False, num_workers=2,
                                persistent_workers=True, pin_memory=True)

class TTADataset(Dataset):
    """每次 __getitem__ 都重新用 tta_tfm 隨機增強，確保 TTA 每次不同"""
    def __init__(self, base_dataset, tta_tfm):
        self.tta_tfm  = tta_tfm
        self.pil_imgs = []
        for f in base_dataset.files:
            self.pil_imgs.append(Image.open(f).convert("RGB"))

    def __len__(self):
        return len(self.pil_imgs)

    def __getitem__(self, idx):
        return self.tta_tfm(self.pil_imgs[idx]), -1

tta_dataset = TTADataset(test_set, tta_tfm)

print(f"測試集大小：{len(test_set)}")
print(f"TTA 次數：{tta_times}")
print(f"參與 Ensemble 的模型數：{n_folds} 折")
print(f"總預測次數（每張圖）：{n_folds} × ({tta_times}+1) = {n_folds*(tta_times+1)}")

In [ ]:
all_logits = torch.zeros(len(test_set), 11)

for k in range(n_folds):
    ckpt_path = f"{_exp_name}_fold{k}.ckpt"
    if not os.path.exists(ckpt_path):
        print(f"找不到 {ckpt_path}，跳過")
        continue

    model_k = nn.DataParallel(Classifier()).to(device)
    model_k.module.load_state_dict(torch.load(ckpt_path))
    model_k.eval()
    print(f"\nFold {k+1} 模型載入完成")

    # 1. 一般預測（不做增強）
    print(f"  Fold {k+1} 一般預測...")
    with torch.no_grad():
        for i, (imgs, _) in enumerate(tqdm(test_loader_normal)):
            logits = model_k(imgs.to(device)).cpu()
            start  = i * batch_size
            end    = start + logits.size(0)
            all_logits[start:end] += logits

    # 2. TTA 預測（每次重新建 DataLoader 確保隨機增強不同）
    for t in range(tta_times):
        print(f"  Fold {k+1} TTA {t+1}/{tta_times}...")
        tta_loader_t = DataLoader(
            tta_dataset, batch_size=batch_size,
            shuffle=False, num_workers=0, pin_memory=True
        )
        with torch.no_grad():
            for i, (imgs, _) in enumerate(tqdm(tta_loader_t)):
                logits = model_k(imgs.to(device)).cpu()
                start  = i * batch_size
                end    = start + logits.size(0)
                all_logits[start:end] += logits

prediction = all_logits.argmax(dim=1).numpy().tolist()
print(f"\n預測完成！共 {len(prediction)} 筆")

# Generate submission.csv

In [ ]:
def pad4(i):
    return str(i).zfill(4)

df = pd.DataFrame()
df["Id"]       = [pad4(i) for i in range(len(test_set))]
df["Category"] = prediction
df.to_csv("submission.csv", index=False)
print("submission.csv 已儲存！")
print(df.head(10))

from IPython.display import FileLink
FileLink('submission.csv')

# Q1. Augmentation Implementation（GradeScope 提交版）

In [ ]:
train_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomCrop(128, padding=16),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

# Q2. t-SNE 視覺化

**layer index：**
- mid layer = `model.cnn[:8]`（Block 1~2 後）
- top layer = `model.cnn[:19]`（全部 6 個 Block 後）

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm

model_vis = nn.DataParallel(Classifier()).to(device)
model_vis.module.load_state_dict(torch.load(f"{_exp_name}_fold0.ckpt"))
model_vis.eval()

print("CNN sub-layers:")
for i, layer in enumerate(model_vis.module.cnn):
    print(f"  [{i}] {layer}")

In [ ]:
vis_valid_set    = FoodDataset(f"{DATA_DIR}/valid", tfm=test_tfm)
vis_valid_loader = DataLoader(vis_valid_set, batch_size=64,
                               shuffle=False, num_workers=2,
                               persistent_workers=True, pin_memory=True)

def visualize_tsne(model, loader, layer_index, title, device):
    features_list, labels_list = [], []
    for imgs, lbls in tqdm(loader, desc=f"Extracting [{title}]"):
        with torch.no_grad():
            feat = model.module.cnn[:layer_index](imgs.to(device))
            feat = feat.view(feat.size(0), -1)
        features_list.extend(feat.cpu().numpy())
        labels_list.extend(lbls.numpy())

    features = np.array(features_list)
    labels   = np.array(labels_list)
    print(f"[{title}] feature 維度：{features.shape}")
    print(f"[{title}] 執行 t-SNE 中（可能需要數分鐘）...")

    features_2d = TSNE(n_components=2, init='pca',
                       random_state=42, perplexity=30).fit_transform(features)

    plt.figure(figsize=(10, 8))
    colors = cm.rainbow(np.linspace(0, 1, 11))
    for label_id in np.unique(labels):
        mask = (labels == label_id)
        plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
                    label=f"Class {label_id}", color=colors[label_id],
                    s=5, alpha=0.7)
    plt.legend(markerscale=3, loc='best')
    plt.title(f"t-SNE Visualization ({title})")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.tight_layout()
    fname = f"tsne_{title.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"圖片已儲存：{fname}")

# mid layer（Block 1~2 後，index=8）
visualize_tsne(model_vis, vis_valid_loader, layer_index=8,
               title="Mid Layer (Block 1-2)", device=device)

# top layer（全部 6 個 Block 後，index=19）
visualize_tsne(model_vis, vis_valid_loader, layer_index=19,
               title="Top Layer (Block 1-6)", device=device)